# 🔍 02 — Dataset Validation
## SmartMine Vision AI · Stage 1: PPE Detection

---

### Objectives
1. Detect **corrupted or unreadable images**.
2. Find **missing label files**.
3. Identify **empty annotation files** (background images).
4. Flag **out-of-bounds bounding boxes** (YOLO coords outside [0,1]).
5. Detect **duplicate images** via MD5 hash.
6. Generate a **machine-readable validation report** saved to `docs/research/`.

> A clean dataset is the foundation of a reliable model.
> We validate **all 5,785 images** — no sampling.

## 1. Setup

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Resolve project root by walking up until 'src/' is found
_cwd = Path().resolve()
PROJECT_ROOT = _cwd
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import hashlib, json
from collections import Counter
from datetime import datetime

import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.ppe_detection.utils import (
    ensure_dirs,
    TRAIN_IMAGES, TRAIN_LABELS,
    VALID_IMAGES, VALID_LABELS,
    TEST_IMAGES,  TEST_LABELS,
    DOCS_DIR, OUTPUTS_DIR,
)
ensure_dirs()
sns.set_theme(style="whitegrid")
print("Setup complete.")

## 2. Validation Logic

We check four independent properties for each image:

| Check | Pass | Fail |
|---|---|---|
| **Readable** | cv2.imread returns array | Returns None or empty |
| **Label exists** | .txt file alongside image | Missing label |
| **Annotation format** | 5 floats per line | Wrong column count |
| **Bbox bounds** | All values in [0,1] | Any value outside range |
| **Duplicate** | Unique MD5 hash | Hash seen before |

Empty labels are reported separately — they are valid **background images**,
not errors.

In [ ]:
def check_image(p: Path) -> str:
    img = cv2.imread(str(p))
    if img is None:      return "unreadable"
    if img.size == 0:    return "empty_array"
    return None

def check_label(p: Path) -> list:
    issues = []
    lines  = [l for l in p.read_text().splitlines() if l.strip()]
    if not lines:
        return ["background_image"]
    for i, line in enumerate(lines):
        parts = line.strip().split()
        if len(parts) != 5:
            issues.append(f"line_{i}:bad_format")
            continue
        try:
            _, cx, cy, w, h = map(float, parts)
        except ValueError:
            issues.append(f"line_{i}:not_numeric")
            continue
        if not (0.0 <= cx <= 1.0 and 0.0 <= cy <= 1.0
                and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
            issues.append(f"line_{i}:out_of_bounds")
    return issues

def md5(path: Path, chunk: int = 65536) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        while data := f.read(chunk):
            h.update(data)
    return h.hexdigest()

print("Validation functions defined.")

## 3. Run Full Validation — All Splits

In [ ]:
splits = [
    ("train", TRAIN_IMAGES, TRAIN_LABELS),
    ("valid", VALID_IMAGES, VALID_LABELS),
    ("test",  TEST_IMAGES,  TEST_LABELS),
]

records:     list = []
seen_hashes: dict = {}

for split_name, img_dir, lbl_dir in splits:
    images = sorted(img_dir.glob("*.jpg")) + sorted(img_dir.glob("*.png"))
    print(f"  Checking {split_name}: {len(images)} images...", end=" ")

    for img_path in images:
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        record   = {"split": split_name, "file": img_path.name,
                    "has_label": lbl_path.exists(), "issues": []}

        # Image integrity
        err = check_image(img_path)
        if err:
            record["issues"].append(err)

        # Label checks
        if not lbl_path.exists():
            record["issues"].append("missing_label")
        else:
            record["issues"].extend(check_label(lbl_path))

        # Duplicate detection
        h = md5(img_path)
        if h in seen_hashes:
            record["issues"].append(f"duplicate_of:{seen_hashes[h]}")
        else:
            seen_hashes[h] = img_path.name

        record["n_issues"]    = len([i for i in record["issues"]
                                      if i != "background_image"])
        record["background"]  = "background_image" in record["issues"]
        records.append(record)

    print("done")

val_df = pd.DataFrame(records)
print(f"\nTotal validated: {len(val_df):,} files")

## 4. Results Summary

In [ ]:
# Split-level summary
summary = val_df.groupby("split").agg(
    total    = ("file", "count"),
    with_label = ("has_label", "sum"),
    background = ("background", "sum"),
    with_issues = ("n_issues", lambda x: (x > 0).sum()),
).rename(columns={"with_label": "has_label"})

print("VALIDATION SUMMARY")
print("=" * 60)
print(summary.to_string())
print()

# Overall issue type counts (excluding background_image)
real_issues = [
    i for record in records
    for i in record["issues"]
    if i != "background_image"
]
issue_counts = Counter(real_issues)

if issue_counts:
    print("REAL ISSUES FOUND:")
    for issue, cnt in issue_counts.most_common():
        print(f"  {issue:<40} {cnt}")
else:
    print("✅  No real issues found — dataset is clean.")

bg_count = val_df["background"].sum()
print(f"\nBackground images (empty labels, valid): {bg_count}")

## 5. Background Images Analysis

In [ ]:
bg_df = val_df[val_df["background"]].copy()
print(f"Background images by split:")
print(bg_df.groupby("split")["file"].count().to_string())
print()
print("Background images are intentional — they teach the model to not")
print("fire false positives on scenes with no PPE or persons.")

# Show a few background image filenames
print("\nSamples:")
for _, row in bg_df.head(5).iterrows():
    print(f"  [{row['split']}] {row['file']}")

## 6. Problem Files Detail

In [ ]:
problem_df = val_df[val_df["n_issues"] > 0][["split", "file", "issues"]]
if len(problem_df) == 0:
    print("No problematic files.")
else:
    print(f"Files with real issues: {len(problem_df)}")
    print(problem_df.to_string(index=False))

## 7. Validation Report — Save

In [ ]:
report = {
    "generated_at": datetime.now().isoformat(),
    "dataset": "SmartMine Unified Dataset",
    "total_files": len(val_df),
    "splits": {
        s: {
            "images":      int(summary.loc[s, "total"]),
            "with_label":  int(summary.loc[s, "has_label"]),
            "background":  int(summary.loc[s, "background"]),
            "with_issues": int(summary.loc[s, "with_issues"]),
        }
        for s in summary.index
    },
    "real_issue_counts": dict(issue_counts),
    "verdict": "CLEAN" if not issue_counts else "ISSUES FOUND",
}

report_json = DOCS_DIR / "smartmine_validation_report.json"
report_csv  = DOCS_DIR / "smartmine_validation_detail.csv"

report_json.write_text(json.dumps(report, indent=2))
val_df.to_csv(report_csv, index=False)

print(f"Report (JSON) → {report_json}")
print(f"Detail (CSV)  → {report_csv}")
print(f"\nVerdict: {report['verdict']}")

## 8. Conclusions & Next Steps

> **Draft** — values below reflect results from running this notebook.
> The known dataset state has some label issues (see `docs/research/smartmine_validation_report.json`).

| Result | Value |
|---|---|
| Dataset integrity | Run notebook to verify |
| Background images | Run notebook to verify |
| Duplicate images | Run notebook to verify |
| Corrupt images | Run notebook to verify |
| Missing labels | Run notebook to verify |
| Out-of-bounds bboxes | Run notebook to verify |

**Next:** `03_training_yolo.ipynb` — fine-tune YOLOv8n on the validated dataset.